[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/rubiks_cube/rubiks_cube.ipynb)

# Solving a Rubik's Cube

This example scrambles a Rubik's cube and solves it. The cube's 54 stickers move only by quarter turns of its six faces, and a scramble of ten random quarter turns is solved by searching outward from both the scrambled and the solved cube at once, every state a quarter turn further each time, until the two searches reach a common state.

Each sticker is a point, and a quarter turn of a face is a rotor, `1 + plane` for the plane the face turns in. Its sandwich doubles lengths, so as a map on points halved, `((1 + plane) >> Point) * 0.5`, it turns by exactly a quarter with coefficients of nothing but 0 and ±1. Placed at integer coordinates, the stickers keep integer coordinates through any number of turns, so states compare exactly. The twelve turn maps form one batch, and applying the batch to a batch of states turns every state every way at once: the search's whole next level is one application.

In Singmaster's notation the faces along +x, -x, +y, -y, +z and -z read as R, L, B, F, U and D, each turned clockwise seen from outside, and the cube's states form the Rubik's cube group, a subgroup of the permutations of the 48 moving stickers.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import stack
from examples.animation import show_animation
from examples.geometry.rubiks_cube import core, render
from examples.geometry.rubiks_cube.core import Plane, Point, animate, codes, faces, mv, planes, solve, solved_cube, turns

np.set_printoptions(precision=4, suppress=True)

## 1. Quarter turns as maps

A face is a plane, a side times an axis, and a sticker belongs to a face's layer when it reaches past 1 along it, `face & sticker > 1`. A quarter turn turns the stickers of its layer and leaves the rest. Each turn's map has coefficients 0 and ±1 only.

In [ ]:
def turned(stickers: Point, turn: Point, face: Plane) -> Point:
    """The stickers after a quarter turn of a face: those in its layer turned, the rest left."""
    layer = (face & stickers) > 1                                          # [...] boolean
    return stickers + layer * (turn(stickers) - stickers)                  # [...] Point


def frontier(stickers: Point) -> Point:
    """Every quarter turn, each way round each face, applied to each of the given states:
    `[2, 2, 3, states, 54]`."""
    return turned(stickers, turns[..., None, None], faces[..., None, None])


solved, corners, colours = solved_cube()                                       # [54] Point, [54, 4] Point, [54] int

In [ ]:
print("a quarter turn of the face along +z, a map on points:\n", turns[0, 0, 2].coefficient_table())

## 2. Scrambling and solving

Ten random quarter turns, none undoing the one before, scramble the cube. The search widens the states reachable from the scrambled cube and from the solved one by turns, alternately, each level one application of the twelve turns to every state of the last, keeping only states not seen before; where the two meet, the turns from the scramble to the meeting state and back from there to the solved cube are the solution.

In [ ]:
# The scramble's quarter turns and its seed, and how many turns the search goes from each side.
n_scramble, seed, depth = 10, 4, 5

In [ ]:
def random_turns(rng: np.random.Generator) -> Iterator[np.ndarray]:
    """Random quarter turns without end, each a direction, a side and an axis, none undoing the one
    before."""
    last = rng.integers((2, 2, 3))
    yield last
    while True:
        move = rng.integers((2, 2, 3))
        if (1 - move[0], *move[1:]) != tuple(last):
            last = move
            yield move


def scramble(stickers: core.Point, moves: np.ndarray) -> core.Point:
    """The stickers after the quarter turns `[turns, 3]`, one after another."""
    for direction, side, axis in moves:
        stickers = core.turned(stickers, core.turns[direction, side, axis], core.faces[side, axis])   # [54] Point
    return stickers


moves = np.array(list(islice(random_turns(np.random.default_rng(seed)), n_scramble)))   # [turns, 3] int
scrambled = scramble(solved, moves)                                            # [54] Point
solution = solve(scrambled, solved, depth)                                     # [turns, 3] int

In [ ]:
print("scramble, direction side axis:\n", moves)
print("solution:\n", solution)

## 3. The scramble and the solution, turned

The animation turns each layer smoothly, by the rotor of a growing fraction of the quarter, `(plane * (-angle / 2)).exp()`, through the scramble and then the solution, back to the solved cube.

In [ ]:
# The frames per quarter turn, and a frame's duration in milliseconds.
n_steps, frame_duration = 6, 60

In [ ]:
states = animate(solved, corners, np.concatenate([moves, solution]), n_steps)
show_animation(render.animate(states, colours), "rubiks_cube", frame_duration)

In [ ]:
# checks
# The solution takes the scrambled cube back exactly, in no more turns than the scramble.
assert (codes(scramble(scrambled, solution)[None]) == codes(solved[None])).all()
assert len(solution) <= n_scramble